# 🎗️ Intelligence Lab – ECE | NOTEBOOK A : DÉMO GUIDÉE
## Breast Cancer Wisconsin – Diagnostic de tumeurs (Cancéreuse / Non-cancéreuse)

---

### 🧠 Contexte Métier

Le dataset **Breast Cancer Wisconsin (Diagnostic)** contient des données issues d'analyses cytologiques de biopsies mammaires.  
Chaque observation correspond à une tumeur décrite par **~30 mesures numériques** calculées à partir d'images de cellules.  
L'objectif est de **prédire si une tumeur est cancéreuse (1) ou non-cancéreuse (0)** à partir de ces mesures.  
C'est un problème de **classification binaire** classique, idéal pour illustrer un pipeline de Machine Learning complet.  
Le dataset est disponible directement via `scikit-learn`, ce qui en fait un point d'entrée parfait pour la démo.

### 📋 Description des variables (features)

Les 30 features sont calculées à partir d'images de cellules et décrivent la forme et la texture des noyaux cellulaires. Elles sont organisées en 3 groupes (moyenne, erreur standard, valeur maximale) pour 10 mesures de base :

| Nom original | Nom français | Description |
|---|---|---|
| mean radius | Rayon moyen | Rayon moyen du noyau cellulaire |
| mean texture | Texture moyenne | Variation de l'intensité en niveaux de gris |
| mean perimeter | Périmètre moyen | Périmètre moyen du noyau |
| mean area | Aire moyenne | Surface moyenne du noyau |
| mean smoothness | Lissé moyen | Variation locale du rayon |
| mean compactness | Compacité moyenne | Périmètre² / aire – 1.0 |
| mean concavity | Concavité moyenne | Sévérité des parties concaves du contour |
| mean concave points | Points concaves moyens | Nombre de portions concaves du contour |
| mean symmetry | Symétrie moyenne | Symétrie du noyau |
| mean fractal dimension | Dimension fractale moyenne | Complexité du contour cellulaire |

> 💡 Les préfixes `error` (erreur standard) et `worst` (valeur maximale) s'appliquent aux mêmes 10 mesures, pour un total de **30 features**.

### 🎯 Objectif Pédagogique

Au fil de ce notebook, vous allez :

1. ✅ **Vérifier rapidement la qualité** du dataset (types, NaN, distributions)
2. 📊 **Visualiser les données** pour comprendre les liens entre features et cible
3. 🛠️ **Feature Engineering** : comprendre quelles variables sont construites et lesquelles sont redondantes
4. 🔧 **Préparer les features** (normalisation entre 0 et 1, PCA)
5. ✂️ **Faire un split 70% / 15% / 15%** (Entraînement / Validation / Test)
6. 🤖 **Entraîner un modèle de classification binaire** et analyser ses résultats

---

### 🌍 5 Sources d'Open Data à connaître

| # | Source | Description |
|---|--------|-------------|
| 🏆 | [Kaggle](https://www.kaggle.com/datasets) | Compétitions + datasets variés |
| 🔍 | [Google Dataset Search](https://datasetsearch.research.google.com/) | Moteur de recherche de datasets |
| 🇫🇷 | [Data.gouv.fr](https://www.data.gouv.fr/) | Données publiques françaises |
| 🤗 | [HuggingFace Datasets](https://huggingface.co/datasets) | NLP, vision, tabular, multi-modal |
| 📂 | [UCI Repository](https://archive.ics.uci.edu/ml/) | Référence académique classique |

---
## 📦 1. Chargement & Aperçu des Données

Le dataset est chargé directement depuis `sklearn.datasets`.  
Il contient **569 observations** et **30 features numériques** + 1 colonne cible `target` (0 = Non-cancéreuse, 1 = Cancéreuse).  
Les features sont toutes numériques et le dataset est **exempt de valeurs manquantes** — parfait pour se concentrer sur le pipeline plutôt que le nettoyage.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# ── Palette ECE ──────────────────────────────────────────────────────────────
ORANGE = "#E69635"
TEAL   = "#306F78"
PALETTE = [TEAL, ORANGE]
sns.set_theme(style="whitegrid", palette=PALETTE)

# ── Dictionnaire de traduction des features ───────────────────────────────────
TRADUCTION = {
    'mean radius':             'Rayon moyen',
    'mean texture':            'Texture moyenne',
    'mean perimeter':          'Périmètre moyen',
    'mean area':               'Aire moyenne',
    'mean smoothness':         'Lissé moyen',
    'mean compactness':        'Compacité moyenne',
    'mean concavity':          'Concavité moyenne',
    'mean concave points':     'Points concaves moyens',
    'mean symmetry':           'Symétrie moyenne',
    'mean fractal dimension':  'Dim. fractale moyenne',
    'radius error':            'Erreur rayon',
    'texture error':           'Erreur texture',
    'perimeter error':         'Erreur périmètre',
    'area error':              'Erreur aire',
    'smoothness error':        'Erreur lissé',
    'compactness error':       'Erreur compacité',
    'concavity error':         'Erreur concavité',
    'concave points error':    'Erreur points concaves',
    'symmetry error':          'Erreur symétrie',
    'fractal dimension error': 'Erreur dim. fractale',
    'worst radius':            'Rayon maximal',
    'worst texture':           'Texture maximale',
    'worst perimeter':         'Périmètre maximal',
    'worst area':              'Aire maximale',
    'worst smoothness':        'Lissé maximal',
    'worst compactness':       'Compacité maximale',
    'worst concavity':         'Concavité maximale',
    'worst concave points':    'Points concaves maximaux',
    'worst symmetry':          'Symétrie maximale',
    'worst fractal dimension': 'Dim. fractale maximale',
}

# ── Chargement ───────────────────────────────────────────────────────────────
data_bc = load_breast_cancer(as_frame=True)
df = data_bc.frame
df = df.rename(columns=TRADUCTION)
df['target'] = df['target'].map({0: 1, 1: 0})  # 1 = Cancéreuse, 0 = Non-cancéreuse

print(f"✅ Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"   Cible : 0 = Non-cancéreuse ({(df['target']==0).sum()}) | 1 = Cancéreuse ({(df['target']==1).sum()})")

In [ ]:
# Aperçu des premières lignes
df.head()

In [ ]:
# Statistiques descriptives
df.describe().round(2)

---
## 🔍 2. Aperçu & Qualité des Données

Avant tout traitement, on vérifie systématiquement :
- **Valeurs manquantes** : y a‑t‑il des NaN ? Combien ? Dans quelles colonnes ?
- **Types** : toutes les colonnes sont-elles du bon type ?
- **Distribution de la cible** : équilibre des classes ?

Pour ce dataset, on s'attend à peu de surprises — mais la rigueur reste essentielle ! 🧪

In [ ]:
# ── Valeurs manquantes ───────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_with_nan = missing[missing > 0]

if missing_with_nan.empty:
    print("✅ Aucune valeur manquante détectée dans le dataset. Parfait !")
else:
    print(f"⚠️ {len(missing_with_nan)} colonnes avec des valeurs manquantes :")
    display(missing_with_nan)

In [ ]:
# ── Distribution de la cible ─────────────────────────────────────────────────
class_counts = df['target'].value_counts().rename({0: 'Non-cancéreuse', 1: 'Cancéreuse'})

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=[TEAL, ORANGE], ax=ax, edgecolor='white', width=0.5)
ax.set_title('Distribution des tumeurs', fontsize=13, fontweight='bold')
ax.set_xlabel('Type de tumeur')
ax.set_ylabel("Nombre d'observations")
ax.set_xticklabels(['Non-cancéreuse', 'Cancéreuse'], rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width()/2, p.get_height()+5),
                ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distributions de quelques features clés ──────────────────────────────────
features_to_plot = [
    'Rayon moyen', 'Texture moyenne', 'Périmètre moyen',
    'Aire moyenne', 'Concavité moyenne', 'Symétrie moyenne'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    for cls, color, label in zip([0, 1], PALETTE, ['Non-cancéreuse', 'Cancéreuse']):
        axes[i].hist(df[df['target']==cls][feat], bins=30, alpha=0.6,
                     color=color, label=label, edgecolor='white')
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valeur')
    axes[i].set_ylabel('Fréquence')
    axes[i].legend(fontsize=8)

fig.suptitle('📊 Distributions de 6 features clés par type de tumeur', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 🛠️ 3. Feature Engineering

Le **Feature Engineering** consiste à créer, transformer ou sélectionner les variables pour améliorer les performances du modèle.  
Dans ce dataset, le travail a déjà été fait en amont par les chercheurs — mais il est important de le comprendre.

### 3.1 🔬 Des features déjà construites à partir de mesures brutes

Plusieurs variables de ce dataset ne sont **pas des mesures directes** mais ont été **calculées** à partir d'autres mesures :

| Variable construite | Formule | Pourquoi c'est du Feature Engineering |
|---|---|---|
| **Compacité** | Périmètre² ÷ Aire − 1.0 | Combine 2 mesures pour capturer la forme globale |
| **Concavité** | Profondeur des creux du contour | Dérivée du contour cellulaire, non mesurée directement |
| **Dimension fractale** | Complexité du bord cellulaire | Mesure mathématique abstraite dérivée du contour |

> 💡 La **compacité** est un exemple classique : un cercle parfait a une compacité de 0, tandis qu'une forme irrégulière (comme une tumeur maligne) aura une compacité bien plus élevée. C'est exactement le type de feature qu'un expert métier construirait à la main.

### 3.2 📐 Relation entre Rayon, Périmètre et Aire

Ces trois variables sont **fortement liées mathématiquement** (pour un cercle : Périmètre = 2π×Rayon, Aire = π×Rayon²).  
→ Elles apportent donc une **information redondante** au modèle. On va vérifier cela avec les corrélations.

### 3.3 🗑️ Suppression des variables trop redondantes

Quand deux variables sont très fortement corrélées (> 0.95), l'une d'elles n'apporte presque rien de plus.  
On va identifier et **supprimer les colonnes les plus redondantes** pour simplifier le modèle.

In [ ]:
# ── Vérification de la corrélation Rayon / Périmètre / Aire ──────────────────
trio = ['Rayon moyen', 'Périmètre moyen', 'Aire moyenne']
corr_trio = df[trio].corr().round(3)

print("🔗 Corrélations entre Rayon, Périmètre et Aire :")
print(corr_trio)
print()
print("→ Ces trois variables sont quasi-identiques du point de vue de l'information.")
print("  En Feature Engineering, on pourrait n'en garder qu'une seule.")

# ── Visualisation de la relation Rayon → Aire ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls, color, label in zip([0, 1], PALETTE, ['Non-cancéreuse', 'Cancéreuse']):
    mask = df['target'] == cls
    axes[0].scatter(df[mask]['Rayon moyen'], df[mask]['Périmètre moyen'],
                    alpha=0.5, color=color, s=20, label=label)
    axes[1].scatter(df[mask]['Rayon moyen'], df[mask]['Aire moyenne'],
                    alpha=0.5, color=color, s=20, label=label)

axes[0].set_xlabel('Rayon moyen')
axes[0].set_ylabel('Périmètre moyen')
axes[0].set_title('Rayon vs Périmètre (quasi-identiques)', fontweight='bold')
axes[0].legend()

axes[1].set_xlabel('Rayon moyen')
axes[1].set_ylabel('Aire moyenne')
axes[1].set_title('Rayon vs Aire (relation quadratique)', fontweight='bold')
axes[1].legend()

fig.suptitle('🛠️ Feature Engineering – Redondance entre Rayon, Périmètre et Aire',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Suppression des colonnes les plus redondantes ────────────────────────────
# On identifie les paires de features avec corrélation > 0.95
X_all = df.drop(columns=['target'])
corr_matrix = X_all.corr().abs()

# Matrice triangulaire supérieure
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Colonnes à supprimer (corrélation > 0.95)
cols_to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f"🗑️  {len(cols_to_drop)} variable(s) supprimée(s) car trop corrélées (> 0.95) :")
for c in cols_to_drop:
    max_corr_col = upper[c][upper[c] > 0.95].idxmax()
    print(f"   • '{c}' → corrélée à '{max_corr_col}' ({upper[c][max_corr_col]:.2f})")

# Dataset nettoyé
df_clean = df.drop(columns=cols_to_drop)
print(f"\n✅ Dataset après suppression : {df_clean.shape[1]-1} features (au lieu de {X_all.shape[1]})")

---
## 🔧 4. Nettoyage Léger & Préparation

Le travail de préparation ici se concentre sur :
1. **Séparation** `X` (features) / `y` (cible), en utilisant le dataset nettoyé par le Feature Engineering
2. **Normalisation entre 0 et 1** avec `MinMaxScaler` — chaque feature est ramenée dans l'intervalle [0, 1]

> 💡 **Pourquoi normaliser ?** Les features ont des échelles très différentes (rayon ~6–28 vs aire ~143–2501). Normaliser entre 0 et 1 permet à tous les attributs de contribuer de façon équitable au modèle.

In [ ]:
# ── Séparation X / y ─────────────────────────────────────────────────────────
X = df_clean.drop(columns=['target'])
y = df_clean['target']

feature_names = X.columns.tolist()
print(f"X : {X.shape} | y : {y.shape}")
print(f"Features utilisées : {len(feature_names)} variables numériques")

In [ ]:
# ── Normalisation entre 0 et 1 ────────────────────────────────────────────────
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=feature_names
)

# On choisit deux features garanties présentes après le Feature Engineering
ex1, ex2 = 'Rayon moyen', 'Texture moyenne'

print("✅ Normalisation effectuée (toutes les features sont maintenant entre 0 et 1).")
print(f"\nExemple — {ex1} AVANT normalisation :")
print(f"  Min = {X[ex1].min():.2f} | Max = {X[ex1].max():.2f} | Moyenne = {X[ex1].mean():.2f}")
print(f"\nExemple — {ex1} APRÈS normalisation :")
print(f"  Min = {X_scaled[ex1].min():.2f} | Max = {X_scaled[ex1].max():.2f} | Moyenne = {X_scaled[ex1].mean():.2f}")

In [ ]:
# ── Visualisation avant / après normalisation pour 2 variables ────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for i, (feat, ax_raw, ax_sc) in enumerate(zip(
    [ex1, ex2],
    [axes[0][0], axes[1][0]],
    [axes[0][1], axes[1][1]]
)):
    ax_raw.hist(X[feat], bins=30, color=TEAL, edgecolor='white', alpha=0.85)
    ax_raw.set_title(f'{feat} — AVANT normalisation', fontsize=10, fontweight='bold')
    ax_raw.set_ylabel('Fréquence')

    ax_sc.hist(X_scaled[feat], bins=30, color=ORANGE, edgecolor='white', alpha=0.85)
    ax_sc.set_title(f'{feat} — APRÈS normalisation [0–1]', fontsize=10, fontweight='bold')
    ax_sc.set_ylabel('Fréquence')

fig.suptitle('🔧 Effet de la normalisation (MinMaxScaler) sur 2 features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 5. Visualisation & Compréhension

Les visualisations permettent de :
- Identifier quelles features **séparent bien** les deux classes
- Détecter d'éventuelles **colinéarités** entre variables
- Comprendre la structure du dataset avant l'entraînement

On utilisera :
1. **Graphique en barres horizontales** pour voir le pouvoir discriminant de chaque feature
2. **Heatmap de corrélation** pour identifier les redondances entre variables
3. **PCA 2D** pour visualiser si les classes sont séparables dans un espace réduit

In [ ]:
# ── Pouvoir discriminant des features (barres horizontales) ───────────────────
# Score = écart absolu des moyennes entre classes, divisé par l'écart-type global
# (similaire à un effet de taille de Cohen – plus le score est élevé, plus la feature sépare bien)
mediane_cancer     = X_scaled[y == 1].mean()
mediane_noncancer  = X_scaled[y == 0].mean()
std_global         = X_scaled.std()

scores = ((mediane_cancer - mediane_noncancer).abs() / std_global).sort_values(ascending=True)
top10  = scores.nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

colors = [ORANGE if v == top10.max() else TEAL for v in top10.values]
bars = ax.barh(top10.index, top10.values, color=colors, edgecolor='white', height=0.6)

# Valeurs à droite de chaque barre
for bar, val in zip(bars, top10.values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left', fontsize=9, color='#333333')

ax.set_xlabel('Score de discrimination (écart normalisé entre les deux classes)', fontsize=10)
ax.set_title('🎯 Top 10 des variables les plus discriminantes\n'
             '(plus le score est élevé → plus la variable sépare cancéreux / non-cancéreux)',
             fontsize=12, fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, top10.max() * 1.15)

# Légende
patch_best = mpatches.Patch(color=ORANGE, label='Meilleure variable')
patch_rest = mpatches.Patch(color=TEAL, label='Top 10')
ax.legend(handles=[patch_best, patch_rest], fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

### 🔗 Heatmap de corrélation

La **heatmap de corrélation** représente les liens linéaires entre paires de variables.  
- Une valeur proche de **+1** (bleu foncé) signifie que les deux variables augmentent ensemble → elles sont **redondantes**.
- Une valeur proche de **-1** (orange foncé) signifie qu'elles évoluent en sens inverse.
- Une valeur proche de **0** signifie qu'il n'y a **pas de lien linéaire**.

> 💡 Des features très corrélées entre elles apportent peu d'information supplémentaire au modèle. C'est ce qu'on a corrigé dans l'étape de Feature Engineering.

In [ ]:
# ── Heatmap de corrélation (features "moyennes" seulement pour la lisibilité) ──
mean_cols = [c for c in feature_names if 'moyen' in c.lower() or 'moyenne' in c.lower()]
corr = X[mean_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(h_neg=200, h_pos=30, s=90, l=40, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('🔗 Heatmap de corrélation – Variables "moyennes"',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔬 Analyse en Composantes Principales (PCA)

La **PCA (Analyse en Composantes Principales)** est une technique qui réduit le nombre de dimensions du dataset tout en conservant le maximum d'information.  
Ici, on réduit les features en seulement **2 composantes principales** pour pouvoir **visualiser les données sur un graphique 2D**.

- Chaque point représente une tumeur
- Les couleurs indiquent le type de tumeur (cancéreuse ou non)
- Si les groupes sont bien séparés visuellement → les deux classes sont **distinguables**, ce qui est bon signe pour notre modèle

> 💡 Le pourcentage de variance expliquée indique combien d'information est conservée après réduction. Plus il est élevé, plus la représentation 2D est fidèle à la réalité.

In [ ]:
# ── PCA 2D : visualiser la séparabilité dans un espace réduit ─────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100
print(f"Variance expliquée : Composante 1 = {explained[0]:.1f}% | Composante 2 = {explained[1]:.1f}%")
print(f"Variance totale conservée (2 composantes) : {explained.sum():.1f}%")

fig, ax = plt.subplots(figsize=(8, 6))
for cls, color, label in zip([0, 1], PALETTE, ['Non-cancéreuse', 'Cancéreuse']):
    mask_cls = y == cls
    ax.scatter(X_pca[mask_cls, 0], X_pca[mask_cls, 1],
               c=color, label=label, alpha=0.65, edgecolors='white', linewidths=0.4, s=50)

ax.set_xlabel(f'Composante 1 ({explained[0]:.1f}% de variance)', fontsize=11)
ax.set_ylabel(f'Composante 2 ({explained[1]:.1f}% de variance)', fontsize=11)
ax.set_title('🔬 PCA 2D – Séparabilité des tumeurs cancéreuses vs non-cancéreuses', fontsize=13, fontweight='bold')
ax.legend(title='Type de tumeur', fontsize=10)
plt.tight_layout()
plt.show()

---
## ✂️ 6. Split 70 / 15 / 15 & Entraînement du Modèle

### Stratégie de split

On découpe les données en **3 ensembles** :

| Ensemble | Rôle | Proportion |
|----------|------|------------|
| **Entraînement** | Entraîner le modèle | 70% |
| **Validation** | Régler les hyperparamètres / surveiller le surapprentissage | 15% |
| **Test** | Évaluation finale non biaisée | 15% |

> ⚠️ On utilise `stratify=y` pour garantir que la proportion de chaque classe est respectée dans chaque split.

### Modèle choisi
**Régression Logistique** — rapide, interprétable, robuste sur des données normalisées.

In [ ]:
# ── Split 70 / 15 / 15 ───────────────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y,
    test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50, random_state=42, stratify=y_temp
)

print("✅ Split effectué :")
print(f"   Entraînement : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"   Validation   : {X_val.shape[0]} lignes ({X_val.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"   Test         : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")

In [ ]:
# ── Entraînement : Régression Logistique ─────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
model.fit(X_train, y_train)

acc_test = accuracy_score(y_test, model.predict(X_test))
print(f"🎯 Précision sur le jeu de test : {acc_test*100:.2f}%")

In [ ]:
# ── Dashboard de résultats ────────────────────────────────────────────────────
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
n_total = len(y_test)
n_train = X_train.shape[0]
n_val   = X_val.shape[0]
n_test  = X_test.shape[0]

fig = plt.figure(figsize=(14, 5), facecolor='white')
fig.suptitle('📊 Résultats du modèle – Régression Logistique', fontsize=14, fontweight='bold', y=1.02)

# ── Panneau 1 : Précision du modèle ──────────────────────────────────────────
ax1 = fig.add_subplot(1, 3, 1)
ax1.set_facecolor('white')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')

# Grand chiffre centré
ax1.text(0.5, 0.62, f"{acc_test*100:.1f}%",
         ha='center', va='center', fontsize=52, fontweight='bold', color=TEAL)
ax1.text(0.5, 0.35, "Précision sur le\njeu de test",
         ha='center', va='center', fontsize=12, color='#555555')
ax1.text(0.5, 0.15, f"sur {n_test} tumeurs",
         ha='center', va='center', fontsize=10, color='#888888')

# Barre de progression
bar_bg  = plt.Rectangle((0.05, 0.07), 0.90, 0.06, color='#EEEEEE', transform=ax1.transAxes, clip_on=False)
bar_val = plt.Rectangle((0.05, 0.07), 0.90 * acc_test, 0.06, color=TEAL, transform=ax1.transAxes, clip_on=False)
ax1.add_patch(bar_bg)
ax1.add_patch(bar_val)
ax1.set_title('🎯 Précision', fontsize=12, fontweight='bold', pad=10)

# ── Panneau 2 : Répartition du split ─────────────────────────────────────────
ax2 = fig.add_subplot(1, 3, 2)
ax2.set_facecolor('white')

labels  = ['Entraînement\n70%', 'Validation\n15%', 'Test\n15%']
sizes   = [n_train, n_val, n_test]
colors2 = [TEAL, '#5B9E8A', ORANGE]
wedges, texts = ax2.pie(sizes, labels=labels, colors=colors2,
                         startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
for text in texts:
    text.set_fontsize(9)

# Ajouter le nombre de lignes dans chaque part
for wedge, n in zip(wedges, sizes):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = 0.55 * np.cos(np.radians(angle))
    y = 0.55 * np.sin(np.radians(angle))
    ax2.text(x, y, str(n), ha='center', va='center', fontsize=10,
             fontweight='bold', color='white')

ax2.set_title('✂️ Répartition du split\n(569 observations)', fontsize=12, fontweight='bold', pad=10)

# ── Panneau 3 : Matrice de confusion ─────────────────────────────────────────
ax3 = fig.add_subplot(1, 3, 3)
ax3.set_facecolor('white')
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Non-cancéreuse', 'Cancéreuse']
)
disp.plot(ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title('🔍 Matrice de confusion\n(jeu de test)', fontsize=12, fontweight='bold', pad=10)
ax3.set_xlabel('Prédiction du modèle', fontsize=9)
ax3.set_ylabel('Réalité', fontsize=9)
ax3.tick_params(axis='x', labelsize=8)
ax3.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

---
## 🏁 Conclusion & Bilan

### Ce que nous avons fait dans ce notebook :

| Étape | Action | Résultat |
|-------|--------|----------|
| ✅ Chargement | `load_breast_cancer(as_frame=True)` | 569 observations × 31 colonnes |
| 🔍 Qualité | Vérification NaN, types, distribution | 0 valeurs manquantes |
| 🛠️ Feature Engineering | Suppression des variables redondantes (> 0.95) | Dataset simplifié sans perte |
| 🔧 Préparation | `MinMaxScaler` – normalisation [0, 1] | Données comparables entre elles |
| 📊 Visualisation | Barres discriminantes, heatmap, PCA 2D | Bonne séparabilité des classes |
| ✂️ Split | 70% / 15% / 15% avec stratification | Entraînement / Validation / Test cohérents |
| 🤖 Modèle | Régression Logistique | ~96%+ de précision sur le test |

### Points clés à retenir 🎯

- Le **Feature Engineering** permet de supprimer les redondances et d'améliorer la lisibilité du modèle
- La **normalisation est indispensable** pour que toutes les features contribuent équitablement au modèle
- La **PCA 2D** montre clairement que les deux classes sont bien séparables dans un espace réduit
- Un **split stratifié** garantit la représentativité de chaque classe dans chaque ensemble
- La **matrice de confusion** révèle les types d'erreurs — en médecine, les faux négatifs (tumeur cancéreuse classée non-cancéreuse) sont bien plus coûteux que les faux positifs !

### Pour aller plus loin 🚀
- Tester d'autres modèles : `RandomForestClassifier`, `SVC`, `GradientBoostingClassifier`
- Utiliser la **validation croisée** (`cross_val_score`) pour des estimations plus robustes
- Analyser les **coefficients** de la régression logistique pour identifier les features les plus importantes
- Calculer l'**AUC-ROC** pour une évaluation plus fine sur les données déséquilibrées